# 4. Flash Attention

In [1]:
import numpy as np

def naive_attention(Q, K, V):
    """Standart, basit ve verimsiz attention hesaplaması."""
    print("--- 1) Naif (Standart) Attention Hesaplanıyor ---")
    d_k = Q.shape[-1]
    
    # Tüm matrisler üzerinde tek seferde işlem yapılır
    scores = (Q @ K.T) / np.sqrt(d_k)
    weights = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    weights /= np.sum(weights, axis=-1, keepdims=True)
    output = weights @ V
    
    print("Naif Attention tamamlandı. Sonuç hafızada.\n")
    return output

def flash_attention_detailed_steps(Q, K, V, block_size):
    """
    FlashAttention'ın mantığını, ara adımları detaylı bir şekilde basarak simüle eder.
    BU BİR PERFORMANS OPTİMİZASYONU DEĞİL, KAVRAMSAL BİR SİMÜLASYONDUR.
    """
    print(f"--- 2) FlashAttention (Detaylı Adımlar) Hesaplanıyor (Blok Boyutu: {block_size}) ---")
    
    seq_len, d_k = Q.shape
    output = np.zeros_like(Q)
    
    num_blocks = seq_len // block_size
    
    # Sadece ilk satırın (ilk kelimenin) evrimini takip edeceğiz
    trace_row_index = 0

    # Q matrisini bloklara ayır (Dış Döngü)
    for i in range(num_blocks):
        start_i, end_i = i * block_size, (i + 1) * block_size
        Q_block = Q[start_i:end_i, :]
        
        # Bu Q bloğu için çıktı ve istatistikleri başlat
        O_block = np.zeros_like(Q_block)
        l_block = np.zeros(block_size)
        m_block = np.full(block_size, -np.inf)
        
        print(f"\n[Dış Döngü] Q'nun {i+1}. bloğu işleniyor (Satır {start_i}-{end_i-1})...")
        if start_i <= trace_row_index < end_i:
            print(f"  Takip edilen satır ({trace_row_index}) için başlangıç durumu:")
            print(f"    m (max skor) = {m_block[trace_row_index - start_i]:.2f}, l (norm. faktör) = {l_block[trace_row_index - start_i]:.2f}")
            print(f"    O (çıktı)    = {np.round(O_block[trace_row_index - start_i], 2)}")

        # K ve V matrislerini bloklara ayır (İç Döngü)
        for j in range(num_blocks):
            start_j, end_j = j * block_size, (j + 1) * block_size
            K_block = K[start_j:end_j, :]
            V_block = V[start_j:end_j, :]
            
            # --- SRAM İÇİNDEKİ HESAPLAMALAR ---
            S_ij = (Q_block @ K_block.T) / np.sqrt(d_k)
            
            # Online Softmax güncellemesi
            m_old = m_block
            l_old = l_block
            O_old = np.copy(O_block) # Karşılaştırma için kopyala
            
            m_new = np.maximum(m_old, np.max(S_ij, axis=1))
            P_ij = np.exp(S_ij - m_new[:, np.newaxis])
            exp_m_diff = np.exp(m_old - m_new)
            
            l_new = exp_m_diff * l_old + np.sum(P_ij, axis=1)
            
            # O vektörünü güncelle: Eski değeri yeniden ölçeklendir ve yeni değeri ekle
            O_block = O_old * exp_m_diff[:, np.newaxis]
            O_block += (P_ij @ V_block)
            
            # Bir sonraki döngü için m ve l'yi güncelle
            m_block = m_new
            l_block = l_new
            
            if start_i <= trace_row_index < end_i:
                print(f"  [İç Döngü] K/V'nin {j+1}. bloğu işlendi:")
                print(f"    m güncellendi: {m_old[trace_row_index - start_i]:.2f} -> {m_new[trace_row_index - start_i]:.2f}")
                print(f"    l güncellendi: {l_old[trace_row_index - start_i]:.2f} -> {l_new[trace_row_index - start_i]:.2f}")
                print(f"    O güncellendi: {np.round(O_old[trace_row_index - start_i, :4], 2)} -> {np.round(O_block[trace_row_index - start_i, :4], 2)}")

        # Bu Q bloğu için tüm hesaplamalar bitti, nihai sonucu HBM'e yaz
        output[start_i:end_i, :] = O_block / l_block[:, np.newaxis]

    print("\nTüm bloklar işlendi. FlashAttention tamamlandı.\n")
    return output

if __name__ == '__main__':
    # Parametreler
    SEQUENCE_LENGTH = 128 
    D_MODEL = 64
    BLOCK_SIZE = 32 

    # Rastgele Q, K, V matrisleri oluştur
    np.random.seed(42)
    Q = np.random.randn(SEQUENCE_LENGTH, D_MODEL)
    K = np.random.randn(SEQUENCE_LENGTH, D_MODEL)
    V = np.random.randn(SEQUENCE_LENGTH, D_MODEL)

    # 1. Yöntem: Standart Yaklaşım (Referans sonuç)
    output_naive = naive_attention(Q, K, V)
    
    # 2. Yöntem: FlashAttention Adım Adım Simülasyonu
    output_flash = flash_attention_detailed_steps(Q, K, V, BLOCK_SIZE)
    
    # 3. Sonuçları Karşılaştırma
    print("--- 3) Sonuçların Karşılaştırılması ---")
    are_same = np.allclose(output_naive, output_flash, atol=1e-6)
    
    print(f"Naif Attention ve FlashAttention sonuçları matematiksel olarak aynı mı? -> {are_same}")
    if are_same:
        print("Kanıt: Adım adım ilerleyen 'online' güncelleme mekanizması, tek seferde yapılan hesaplama ile aynı sonucu vermiştir.")
    else:
        print("Hata: Sonuçlar farklı, simülasyonda bir sorun var.")

    print("\nNihai Çıktı Karşılaştırması (ilk satır):")
    print("Naif  :", np.round(output_naive[0, :5], 4))
    print("Flash :", np.round(output_flash[0, :5], 4))

--- 1) Naif (Standart) Attention Hesaplanıyor ---
Naif Attention tamamlandı. Sonuç hafızada.

--- 2) FlashAttention (Detaylı Adımlar) Hesaplanıyor (Blok Boyutu: 32) ---

[Dış Döngü] Q'nun 1. bloğu işleniyor (Satır 0-31)...
  Takip edilen satır (0) için başlangıç durumu:
    m (max skor) = -inf, l (norm. faktör) = 0.00
    O (çıktı)    = [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
  [İç Döngü] K/V'nin 1. bloğu işlendi:
    m güncellendi: -inf -> 1.69
    l güncellendi: 0.00 -> 7.83
    O güncellendi: [0. 0. 0. 0.] -> [-4.61  2.12  0.68 -0.24]
  [İç Döngü] K/V'nin 2. bloğu işlendi:
    m güncellendi: 1.69 -> 1.69
    l güncellendi: 7.83 -> 15.34
    O güncellendi: [-4.61  2.12  0.68 -0.24] -> [-2.96  3.25 -1.54 -1.71]
  [İç Döngü] K/V'nin 3. bloğu işlendi:
    m güncellendi: 1.69 -> 2.01
    l güncellendi: 15.34 -> 17.04
    O güncellendi